# Lecture 5.5 — Building a Triage Agent with Multiple Handoff Targets

**Section 05 — Multi-Agent Orchestration & Guardrails**

In this notebook you will build a support triage system with three specialist
agents, wire up handoffs in both directions, and run a multi-turn conversation
that routes itself dynamically between agents.


## Cell 1: Installing the OpenAI Agents SDK

This notebook uses the `openai-agents` Python package, which provides the
`Agent`, `Runner`, and `handoff` building blocks used throughout this
notebook.

The install below pins a specific version so that the code you see here
behaves the same way every time you run it. If the package is already
present in your current Colab session, this cell finishes almost instantly
and simply confirms the version is available.

In [ ]:
# Pinned for reproducibility. To use the latest version,
# run: pip install openai-agents
# Or substitute your preferred version below.
!pip install openai-agents==0.18.3 -q

## Cell 2: Setting Up Your OpenAI API Key

This notebook calls the OpenAI API, so it needs your API key available as
an environment variable. In Google Colab, the safest way to do this is
Colab Secrets, which keeps the key out of the notebook itself.

**Steps to add your key as a Colab Secret:**

1. Click the key icon in the left sidebar of Colab to open the **Secrets** panel.
2. Click **Add new secret**.
3. Set the name to `OPENAI_API_KEY`.
4. Paste your API key as the value.
5. Toggle **Notebook access** on for this notebook.

The cell below reads that secret with `userdata.get()` and writes it into
`os.environ`, which is where the OpenAI client library looks for it
automatically.

**Local users:** if you are running this outside Colab, set the
`OPENAI_API_KEY` environment variable in your terminal before starting
Jupyter instead of using this cell, for example `export OPENAI_API_KEY="sk-..."`
on macOS or Linux.

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Cell 3: Declaring the Model Name

Every agent in this notebook passes `model=MODEL_NAME` instead of a
hardcoded model string. Declaring it once here means you can switch every
agent in the notebook to a different model by changing a single line.

| Variable | Value | Purpose |
|---|---|---|
| `MODEL_NAME` | `"gpt-5.4-mini"` | Passed to every `Agent(...)` in this notebook |

See the current list of available models at
[platform.openai.com/docs/models](https://platform.openai.com/docs/models).

In [ ]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

## Cell 4: Imports

This cell brings in everything the notebook needs, grouped by where each
name comes from.

| Import | Source | Used for |
|---|---|---|
| `Reasoning` | `openai.types.shared` | Configuring reasoning effort on `ModelSettings` |
| `Agent` | `agents` | Defining every agent in this notebook |
| `HandoffOutputItem` | `agents` | Identifying handoff events inside `new_items` |
| `ItemHelpers` | `agents` | Extracting plain text from message items |
| `MessageOutputItem` | `agents` | Identifying agent text replies inside `new_items` |
| `ModelSettings` | `agents` | Tuning reasoning effort and verbosity per agent |
| `Runner` | `agents` | Executing an agent with `await Runner.run(...)` |
| `TResponseInputItem` | `agents` | Typing the manually built conversation input list |
| `function_tool` | `agents` | Turning a plain Python function into a tool |
| `handoff` | `agents` | Building a handoff with a custom tool name |
| `RECOMMENDED_PROMPT_PREFIX` | `agents.extensions.handoff_prompt` | A prefix added to every agent's instructions |

Notice that `RECOMMENDED_PROMPT_PREFIX` does not come from the top-level
`agents` package. It lives in its own submodule,
`agents.extensions.handoff_prompt`, because it is an optional extension
rather than a core primitive.

In [ ]:
from openai.types.shared import Reasoning

from agents import (
    Agent,
    HandoffOutputItem,
    ItemHelpers,
    MessageOutputItem,
    ModelSettings,
    Runner,
    TResponseInputItem,
    function_tool,
    handoff,
)
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX

## Cell 5: A Three-Way Triage: Routing by Language

Before building a richer domain, this cell sets up the simplest possible
triage shape: one triage agent with three specialists, each speaking a
different language. The triage agent has no tools of its own. Its only job
is to read the request and hand off to whichever specialist fits.

Two details matter here:

| Detail | Why it matters |
|---|---|
| `handoff_description` on each specialist | This is the text the triage agent's model reads when deciding where to route. Without it, the model only has the agent's name to go on. |
| `RECOMMENDED_PROMPT_PREFIX` prepended to every agent's instructions | This is the SDK's own recommended prefix for any agent that participates in handoffs. It teaches the model what a handoff is, mechanically, so routing decisions are more reliable. |

Notice that `language_triage.handoffs` is passed a plain list of `Agent`
objects, not `handoff(...)` calls. This is the simplest form a handoff can
take. Later in this notebook you will see the `handoff(...)` function used
directly, which unlocks options like a custom tool name.

Each agent also sets `model_settings=ModelSettings(reasoning=Reasoning(effort="none"), verbosity="low")`.
These are routing and short-reply agents, so a fast, low-effort response is
what you want here.

In [ ]:
french_agent = Agent(
    name="French Agent",
    handoff_description=(
        "Speaks fluent French. "
        "Use for any French-language request."
    ),
    instructions=(
        f"{RECOMMENDED_PROMPT_PREFIX}\n"
        "You only speak French. Respond entirely in French."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

spanish_agent = Agent(
    name="Spanish Agent",
    handoff_description=(
        "Speaks fluent Spanish. "
        "Use for any Spanish-language request."
    ),
    instructions=(
        f"{RECOMMENDED_PROMPT_PREFIX}\n"
        "You only speak Spanish. Respond entirely in Spanish."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

english_agent = Agent(
    name="English Agent",
    handoff_description=(
        "Speaks fluent English. "
        "Use for any English-language request."
    ),
    instructions=(
        f"{RECOMMENDED_PROMPT_PREFIX}\n"
        "You only speak English. Respond entirely in English."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

language_triage = Agent(
    name="Language Triage",
    instructions=(
        f"{RECOMMENDED_PROMPT_PREFIX}\n"
        "Handoff to the appropriate agent based on the "
        "language of the request."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    handoffs=[french_agent, spanish_agent, english_agent],
)

result = await Runner.run(
    language_triage,
    "Bonjour, comment allez-vous?",
)

print("Final output:", result.final_output)
print("Last agent:", result.last_agent.name)

## Cell 6: Function Tools for a Support Domain

The three-way language example above is deliberately simple: no tools, just
routing. A real triage system usually routes to specialists that each own
their own tools. This cell defines two small function tools that the
specialist agents below will use.

| Tool | Purpose |
|---|---|
| `lookup_faq` | Looks up an answer in a small hardcoded FAQ dictionary |
| `update_seat` | "Updates" a seat assignment for a booking confirmation number |

Both are deliberately simplified, hardcoded stand-ins so the notebook stays
focused on handoff behavior rather than on building a real backend. Each
function's docstring `Args:` section is what the SDK turns into the JSON
schema the model sees, so the parameter descriptions matter even though the
functions themselves are simple.

In [ ]:
@function_tool
def lookup_faq(question: str) -> str:
    """Looks up an answer in the FAQ knowledge base.

    Args:
        question: The user's question.
    """
    faqs = {
        "hours": "We are open Monday to Friday, 9am to 6pm.",
        "shipping": (
            "Standard shipping takes 3-5 business days."
        ),
    }
    for key, answer in faqs.items():
        if key in question.lower():
            return answer
    return "No matching FAQ found."


@function_tool
def update_seat(confirmation_number: str, seat: str) -> str:
    """Updates a passenger's seat assignment.

    Args:
        confirmation_number: The booking confirmation number.
        seat: The desired seat number.
    """
    return (
        f"Seat updated to {seat} for confirmation "
        f"{confirmation_number}."
    )

## Cell 7: Two Specialist Agents, Each Owning Its Own Tools

This cell defines the two specialists that will sit underneath a triage
agent: an FAQ agent and a seat booking agent. Each one only receives the
tool relevant to its own domain. The FAQ agent cannot update a seat, and the
seat booking agent cannot look up FAQs. This separation is the actual payoff
of a handoff-based design: each specialist stays narrowly focused, with a
narrowly focused toolset and prompt to match.

Look closely at the last line of each agent's instructions: *"If you cannot
answer, transfer back to the triage agent."* Both specialists already
mention transferring back. But at this point in the notebook, neither
specialist has a handoff back to triage wired up yet, because the triage
agent they would transfer to does not exist yet. The instructions are
already there. The actual handoff comes two cells from now.

In [ ]:
faq_agent = Agent(
    name="FAQ Agent",
    handoff_description=(
        "Answers frequently asked questions using the "
        "FAQ tool."
    ),
    instructions=(
        f"{RECOMMENDED_PROMPT_PREFIX}\n"
        "You are an FAQ agent. Use the lookup_faq tool to "
        "answer questions. Do not rely on your own knowledge. "
        "If you cannot answer, transfer back to the "
        "triage agent."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[lookup_faq],
)

seat_agent = Agent(
    name="Seat Booking Agent",
    handoff_description=(
        "Updates seat assignments for flight bookings."
    ),
    instructions=(
        f"{RECOMMENDED_PROMPT_PREFIX}\n"
        "You are a seat booking agent. Ask for the "
        "confirmation number and desired seat, then use "
        "update_seat. If the question is unrelated, "
        "transfer back to the triage agent."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[update_seat],
)

## Cell 8: Building the Triage Agent with Named Handoff Tools

This cell defines `support_triage`, the agent that decides which specialist
handles an incoming message. Unlike the three-way language example earlier,
this triage agent uses the `handoff(...)` function instead of passing raw
`Agent` objects into `handoffs=[...]`.

| Parameter | What it does |
|---|---|
| `agent` | The specialist this handoff targets, for example `faq_agent` |
| `tool_name_override` | Replaces the default auto-generated tool name (something like `transfer_to_faq_agent` derived from the agent's name) with an explicit one you choose |

Using `tool_name_override` here gives the handoff tools predictable,
explicit names: `transfer_to_faq_agent` and `transfer_to_seat_agent`. That
matters once you start logging or debugging a system with several agents,
since a consistent naming scheme makes the routing trail easier to read at
a glance, which you will see for yourself later in this notebook.

**Why `faq_agent.handoff_description` matters right here.** Neither
`handoff()` call below sets `tool_description_override`. When that
parameter is left unset, the SDK builds the handoff tool's description from
a default template instead:

```
Handoff to the {agent.name} agent to handle the request. {handoff_description}
```

That means the `handoff_description` you set on `faq_agent` and
`seat_agent` in the previous cell is not just a comment for other
developers reading the code. It gets read directly into the tool
description the model sees when deciding whether to call
`transfer_to_faq_agent` or `transfer_to_seat_agent`. If a `handoff()` call
did set `tool_description_override`, that override would replace the
entire default template above, and the agent's `handoff_description` would
be ignored for that specific handoff.

`support_triage` can reference `faq_agent` and `seat_agent` directly because
both were already defined in the previous cell.

In [ ]:
support_triage = Agent(
    name="Support Triage",
    handoff_description=(
        "Routes customer requests to the correct specialist."
    ),
    instructions=(
        f"{RECOMMENDED_PROMPT_PREFIX}\n"
        "You are a helpful triage agent. "
        "Delegate FAQ questions to the FAQ Agent. "
        "Delegate seat requests to the Seat Booking Agent."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    handoffs=[
        handoff(
            faq_agent,
            tool_name_override="transfer_to_faq_agent",
        ),
        handoff(
            seat_agent,
            tool_name_override="transfer_to_seat_agent",
        ),
    ],
)

## Cell 9: Solving the Circular Reference: Handoffs Back to Triage

Here is the problem this cell solves. `support_triage` needs to reference
`faq_agent` and `seat_agent` to hand off to them. But if `faq_agent` and
`seat_agent` also need a handoff back to `support_triage`, one of the three
agents has to be defined before the other two, and Python has no way to
define all three at once with circular references between them.

The fix is `Agent.handoffs`. It is declared as a plain mutable Python list,
not a fixed tuple, so it can be appended to after an agent has already been
constructed. That is exactly what this cell does: `support_triage` was
built last, in the previous cell, since it needed both specialists to
already exist. Now that `support_triage` exists, this cell appends a return
handoff onto each specialist's `handoffs` list.

| Step | Order |
|---|---|
| 1 | Define `faq_agent` and `seat_agent` (previous cells) |
| 2 | Define `support_triage`, referencing both specialists (previous cell) |
| 3 | Append a handoff back to `support_triage` onto each specialist (this cell) |

`tool_name_override="transfer_to_triage_agent"` is used on both return
handoffs, so no matter which specialist is sending the conversation back,
the tool name in the trace is identical and easy to search for.

In [ ]:
# Specialists route back to triage. This must happen AFTER
# support_triage exists, since support_triage did not exist yet
# when faq_agent and seat_agent were first defined.

# TODO (live): faq_agent's return handoff goes here — same pattern as below


seat_agent.handoffs.append(
    handoff(
        support_triage,
        tool_name_override="transfer_to_triage_agent",
    )
)

## Cell 10: Running the First Turn

With all three agents fully wired up, it's time to run a conversation. The
first message asks a question squarely inside the FAQ agent's territory, so
`support_triage` should hand off to `faq_agent` immediately.

Two things to notice in this cell:

- `inputs` is typed as `list[TResponseInputItem]`. This is the same shape
  the SDK uses internally for conversation history, so building it this way
  keeps you compatible with everything the SDK gives back later, including
  `result.to_input_list()` in the next cell.
- `current_agent` starts as `support_triage`, since that is where every new
  conversation in this system should begin.

In [ ]:
current_agent = support_triage
inputs: list[TResponseInputItem] = [
    {"role": "user", "content": "What are your store hours?"}
]

result = await Runner.run(current_agent, inputs)

print("Turn 1 output:", result.final_output)
print("Turn 1 last agent:", result.last_agent.name)

## Cell 11: Running a Second, Off-Topic Turn

This cell continues the same conversation with a message that has nothing
to do with FAQs: a seat change request. The FAQ agent currently holds the
conversation, so watch closely what happens when a question outside its
scope arrives.

Two patterns carry the conversation forward:

- `result.to_input_list()` converts everything that happened in turn one,
  including the handoff, into a plain list of input items. Adding a new
  user message onto the end of that list is how you continue a
  conversation across multiple `Runner.run()` calls.
- `current_agent = result.last_agent` means turn two starts with whichever
  agent turn one actually ended on, which is `faq_agent`, not
  `support_triage`. You never restart at triage on every turn. You continue
  with whoever is currently holding the conversation, and let that agent
  decide whether to hand off again.

If the FAQ agent recognizes the seat request as outside its scope, it
should transfer back to `support_triage`, which should then route to
`seat_agent`. Here's the part worth sitting with: that is two handoffs
happening inside a single `Runner.run()` call, entirely without you
manually picking an agent at any point.

To make that round trip visible right away, this cell also walks
`result2.new_items` and prints just the handoff chain: which agent handed
off to which, in order. You should see two lines, the FAQ agent
transferring back to `support_triage`, then `support_triage` transferring
to `seat_agent`. That is the bidirectional handoff from two cells ago,
actually firing. The next cell builds on this with the full routing trail,
including the messages each agent produced along the way.

In [ ]:
inputs = result.to_input_list() + [
    {
        "role": "user",
        "content": (
            "Actually, can you update my seat to 14A? "
            "Confirmation ABC123."
        ),
    }
]
current_agent = result.last_agent
print("Current Agent:", current_agent.name)

result2 = await Runner.run(current_agent, inputs)

print("Turn 2 output:", result2.final_output)
print("Turn 2 last agent:", result2.last_agent.name)

print("Turn 2 handoff chain:")
for item in result2.new_items:
    if isinstance(item, HandoffOutputItem):
        print(f"  {item.source_agent.name} -> {item.target_agent.name}")

## Cell 12: Reading the Full Routing Trail

`result2.new_items` holds every item generated during turn two, including
items generated by agents that only participated briefly, like the FAQ
agent handing the conversation straight back to triage. The previous cell
already printed the bare handoff chain. This cell goes further: it walks
the same list and prints a clean, readable trail that includes what each
agent actually said along the way, not just who handed off to whom.

| Item type | What it represents | Relevant fields |
|---|---|---|
| `HandoffOutputItem` | A handoff took place | `.source_agent.name`, `.target_agent.name` |
| `MessageOutputItem` | An agent produced a reply the user would see | `.agent.name`, plus its text via `ItemHelpers.text_message_output(item)` |

For a support system handling many customers at once, this is exactly the
kind of trail you would log in production: every routing decision,
attributable to the specific agent that made it, in the order it happened.

In [ ]:
print("=== Full routing trail ===")
for i, item in enumerate(result2.new_items):
    if isinstance(item, HandoffOutputItem):
        print(
            f"  Handoff {i}: {item.source_agent.name} -> "
            f"{item.target_agent.name}"
        )
    elif isinstance(item, MessageOutputItem):
        text = ItemHelpers.text_message_output(item)
        print(f"  [{item.agent.name}] {text[:60]}")

## Cell 13: A Request Neither Specialist Can Handle

This notebook has shown two things so far: a specialist handling a request
squarely in its domain, and a specialist correctly handing an out-of-scope
request back to triage. This cell tests a third, trickier case: a request
that matches neither specialist at all.

Look back at `support_triage`'s instructions from Cell 8. They only cover
two situations: delegate FAQ questions to the FAQ Agent, delegate seat
requests to the Seat Booking Agent. There is no third specialist, and
there is no instruction telling triage what to do if a request is neither.

This cell continues the same conversation, picking up wherever
`result2.last_agent` left off, which is `seat_agent`, and asks something
that has nothing to do with seat bookings or FAQs at all.

Watch for two separate things when you run this. First, `seat_agent`
should recognize this is unrelated to its job and transfer back to
`support_triage`, the same pattern you already saw from `faq_agent` in the
previous turn. Second, once `support_triage` receives it, there is nowhere
left to send it. There is no third handoff this time, since neither
specialist fits. The handoff chain should show one hop, not two, and
`last_agent` should land on `Support Triage` itself rather than a
specialist.

In [ ]:
inputs = result2.to_input_list() + [
    {
        "role": "user",
        "content": "By the way, what's the weather like today?",
    }
]
current_agent = result2.last_agent
print("Current Agent:", current_agent.name)

result3 = await Runner.run(current_agent, inputs)

print("Turn 3 output:", result3.final_output)
print("Turn 3 last agent:", result3.last_agent.name)

print("Turn 3 handoff chain:")
handoffs_this_turn = [
    item for item in result3.new_items if isinstance(item, HandoffOutputItem)
]
if handoffs_this_turn:
    for item in handoffs_this_turn:
        print(f"  {item.source_agent.name} -> {item.target_agent.name}")
else:
    print("  (no handoffs this turn)")

## Cell 14: Design Principles for Triage Agents

This notebook built one specific triage system, but the underlying
decisions generalize to any triage design. The table below summarizes them,
including one lesson straight from the cell you just ran.

| Principle | Why |
|---|---|
| Give every specialist a clear `handoff_description` | Triage relies on it to route correctly |
| Use `RECOMMENDED_PROMPT_PREFIX` on every agent | Improves handoff recognition and behaviour |
| Let specialists route back to triage | Handles the "wrong specialist" case gracefully |
| Keep triage instructions focused only on routing | Don't let triage try to answer questions itself |
| Track `last_agent` across turns | Determines which agent handles the next user message |
| Name handoff tools explicitly with `tool_name_override` | Predictable names make logging and debugging easier |
| Give triage an explicit fallback for unmatched requests | Without one, behaviour is undefined the moment a request fits no specialist, as you just saw in Cell 13 |

This notebook deliberately kept a few things simple. It used the default,
full-history behaviour on every handoff rather than an `input_filter`, ran
everything without streaming, and used only two specialists behind a single
triage agent rather than several layers of routing. Each of those is a
reasonable next thing to explore once this core pattern feels solid.